In [52]:
from owlready2 import *
import owlready2 as owl 
from pprint import pprint


In [1]:
from rdflib import Graph, Namespace, RDF
import networkx as nx
from pyvis.network import Network

# Load ontology
path = r'D:\github_repos\POKIMON\visualization\test\testing.owl'
g = Graph()
g.parse(path)  # or .ttl, .rdf

POKI = Namespace("http://www.POKIMON#")

# Query: Get Algorithm instances and their is_About target
query = """
PREFIX poki: <http://www.POKIMON#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?algorithm ?target
WHERE {
  ?algorithm poki:is_About ?target .
  ?algorithm rdf:type poki:Algorithm .
}
"""

results = g.query(query)

In [13]:
results.__dict__

{'type': 'SELECT',
 'vars': [rdflib.term.Variable('subject'),
  rdflib.term.Variable('predicate'),
  rdflib.term.Variable('object')],
 '_bindings': [],
 '_genbindings': <generator object evalDistinct at 0x000001C083828CF0>,
 'askAnswer': None,
 'graph': None}

In [ ]:
#worls

# Build the graph
G = nx.DiGraph()
for row in results:
    algo = str(row["algorithm"].split("#")[-1])
    target = str(row["target"].split("#")[-1])
    G.add_edge(algo, target, label="is_About")

# Create pyvis network
net = Network(height="600px", width="100%", directed=True)

# Add nodes: Label inside node with smaller font
for node in G.nodes():
    net.add_node(
        node,
        label=node,
        shape="dot",
        size=15,
        font={"size": 12, "color": "#222", "face": "arial", "vadjust": -30},
    )

# Add edges: With label, font size, and arrow direction
for source, target, data in G.edges(data=True):
    net.add_edge(
        source,
        target,
        label=data.get("label", ""),
        arrows="to",
        font={"size": 10, "align": "top"},
        length=250,  # space between nodes
        color="#999"
    )

# Save to HTML
net.write_html("ontology_visualization_final.html")




from pyvis.network import Network

net = Network()
net.add_node(1, label="Node 1", font = {"size": 8, "color": "#222", "face": "arial"})
net.add_node(2, label="Node 2")
net.add_edge(1, 2)

net.write_html("ontology_visualization3.html")


In [ ]:
#works great 
from rdflib import Graph, Namespace, RDF
import networkx as nx
from pyvis.network import Network

# Load ontology
path = r'D:\github_repos\POKIMON\visualization\test\testing.owl'
g = Graph()
g.parse(path)  # or .ttl, .rdf

POKI = Namespace("http://www.POKIMON#")

# Define your class-to-color mapping
color_map = {
    "Algorithm": "#1fb469",  # blue
    "Topic": "#ff7f0e",      # orange
    "Default": "#999999"     # fallback gray
}

# Query that also returns the class/type
query = """
PREFIX poki: <http://www.POKIMON#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?instance ?target ?type
WHERE {
  ?instance poki:is_About ?target .
  ?instance rdf:type ?type .
}
"""

results = g.query(query)

G = nx.DiGraph()
node_classes = {}  # Track types for coloring

for row in results:
    instance = str(row["instance"].split("#")[-1])
    target = str(row["target"].split("#")[-1])
    class_uri = str(row["type"].split("#")[-1])

    G.add_edge(instance, target, label="is_About")
    node_classes[instance] = class_uri
    if target not in node_classes:
        node_classes[target] = "Topic"  # You can infer or assign manually

# Build visualization
net = Network(height="600px", width="100%", directed=True)

for node in G.nodes():
    label = str(node)
    node_type = node_classes.get(node, "Default")
    color = color_map.get(node_type, color_map["Default"])
    
    net.add_node(
        node,
        label=label,  # Ensure label is explicitly set
        shape="dot",
        size=35,
        color=color,
        font={
            "size": 12,
            "color": "#000000" if node_type == "Algorithm" else "#000000",
            "face": "arial",
            "vadjust": -40  # center the label vertically
        }
    )


for source, target, data in G.edges(data=True):
    net.add_edge(
        source,
        target,
        label=data.get("label", ""),
        arrows="to",
        font={"size": 10},
        length=250,
        color="#ccc"
    )

net.write_html("ontology_colored_nodes.html")


In [ ]:
from rdflib import Graph, Namespace, RDF
import networkx as nx
from pyvis.network import Network



from matplotlib import cm
import random


def get_color_for_class(cls):
    if cls not in color_map:
        color_map[cls] = f'#{random.randint(0, 0xFFFFFF):06x}'
    return color_map[cls]

# Then use:
color = get_color_for_class(node_type)

# Define color map dynamically
color_map = {
    "Algorithm": "#1f77b4",
    "Topic": "#ff7f0e",
    "Agent": "#2ca02c",
    "Default": "#999999"
}


# Query
query = """
PREFIX poki: <http://www.POKIMON#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?source ?target ?relation ?sourceType ?targetType
WHERE {
  ?source ?relation ?target .
  ?source rdf:type poki:Algorithm .
  OPTIONAL { ?target rdf:type ?targetType . }
  BIND(poki:Algorithm AS ?sourceType)
  FILTER (isIRI(?target))
}
"""

results = g.query(query)

# Build the graph
G = nx.DiGraph()
node_classes = {}
edge_labels = []

def short(uri):
    return uri.split("#")[-1] if "#" in uri else uri.split("/")[-1]

for row in results:
    src = short(row["source"])
    tgt = short(row["target"])
    rel = short(row["relation"])
    src_type = short(row["sourceType"])
    tgt_type = short(row["targetType"]) if row["targetType"] else "Default"

    # Add edge
    G.add_edge(src, tgt, label=rel)

    # Store class for coloring
    node_classes[src] = src_type
    node_classes[tgt] = tgt_type

# Build pyvis graph
net = Network(height="700px", width="100%", directed=True)

for node in G.nodes():
    node_type = node_classes.get(node, "Default")
    color = color_map.get(node_type, color_map["Default"])

    net.add_node(
        node,
        label=node,
        shape="dot",
        size=20,
        color=color,
        font={"size": 12, "color": "#fff" if color != "#999999" else "#000"}
    )

for source, target, data in G.edges(data=True):
    net.add_edge(
        source,
        target,
        label=data.get("label", ""),
        arrows="to",
        font={"size": 10},
        length=250,
        color="#ccc"
    )

# Save
net.write_html("ontology_dynamic_graph.html")


In [11]:
from rdflib import Graph, Namespace, RDF
import networkx as nx
from pyvis.network import Network


from matplotlib import cm
import random



# Colors for types
color_map = {
    "Algorithm": "#1f77b4",            # blue
    "Planned_Process": "#2ca02c",      # green
    "Action_Specification": "#d62728", # red
    "Unknown": "#888"
}

def get_color_for_class(cls):
    if cls not in color_map:
        print('not in', cls)
        color_map[cls] = f'#{random.randint(0, 0xFFFFFF):06x}'
    return color_map[cls]

def get_node_type(uri):
    types = set()
    for _, _, o in g.triples((uri, RDF.type, None)):
        types.add(short(str(o)))
    if types:
        # Return first type or combine multiple if needed
        return list(types)[0]
    else:
        return "Unknown"

def short(uri):
    if uri is None:
        return "Unknown"
    return uri.split("#")[-1] if "#" in uri else uri.split("/")[-1]





POKI = Namespace("http://www.POKIMON#")

query = """

PREFIX poki: <http://www.POKIMON#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?subject ?predicate ?object
WHERE {
  ?subject rdf:type poki:Algorithm .

  {
    ?subject poki:is_About ?object1 .
    ?object1 rdf:type poki:Planned_Process .
  }
  FILTER EXISTS {
    ?subject poki:has_Part ?object2 .
    ?object2 rdf:type poki:Action_Specification .
  }

  {
    ?subject poki:is_About ?object .
    BIND(poki:is_About AS ?predicate)
  }
  UNION
  {
    ?subject poki:has_Part ?object .
    BIND(poki:has_Part AS ?predicate)
  }
}

"""

results = g.query(query)



In [12]:
results.__dict__

{'type': 'SELECT',
 'vars': [rdflib.term.Variable('subject'),
  rdflib.term.Variable('predicate'),
  rdflib.term.Variable('object')],
 '_bindings': [],
 '_genbindings': <generator object evalDistinct at 0x0000025F5F5A87B0>,
 'askAnswer': None,
 'graph': None}

In [13]:
results

In [17]:


G = nx.DiGraph()
node_types = {}

for row in results:
    subj_uri = row.subject
    pred_uri = row.predicate  # now valid
    obj_uri = row.object      # note just one object variable now

    subj = short(str(subj_uri))
    pred = short(str(pred_uri))
    obj = short(str(obj_uri))

    # Get and save subject type if not already done
    if subj not in node_types:
        node_types[subj] = get_node_type(subj_uri)

    # Add edge and set object type
    G.add_edge(subj, obj, label=pred)
    if obj not in node_types:
        node_types[obj] = get_node_type(obj_uri)



# Visualize
net = Network(height="700px", width="100%", directed=True)


for node in G.nodes():
    node_type = node_types.get(node, "Unknown")
    color = get_color_for_class(node_type)
    net.add_node(
        node,
        label=node,
        shape="dot",
        size=35,
        color=color,
        font={
            "size": 12,
            "color": "#000000" ,
            "face": "arial",
            "vadjust": -50  # center the label vertically
        }
    )

for src, tgt, data in G.edges(data=True):
    net.add_edge(
        src,
        tgt,
        label=data.get("label", ""),
        arrows="to",
        font={"size": 10},
        length=250
    )

net.write_html("full_ontology_graph2.html")


In [ ]:
stop


## generating figures with 

In [2]:
import pylode


In [4]:
input_file_path = r'D:\github_repos\POKIMON\old_version\observations.ttl'

html = pylode.MakeDocco(
    input_data_file=input_file_path,
    outputformat="html",
    profile="ontdoc"
).document()

In [7]:
output_html_file = "ontology_doc.html"
with open(output_html_file, "w", encoding="utf-8") as f:
    f.write(html)

In [ ]:
stop

## adding prefix to owl

In [2]:

def add_prefix_to_ontology(input_path, output_path, prefix):
    """
    Loads an ontology from input_path, renames all classes and properties with the given prefix,
    and saves the updated ontology to output_path.

    :param input_path: Path to the input ontology (.owl file).
    :param output_path: Path to save the updated ontology.
    :param prefix: The prefix to add to all class and property names.
    """
    # Load the ontology
    onto = get_ontology(input_path).load()
    
    # Rename classes
    for cls in onto.classes():
        cls.name = prefix + cls.name

    # Rename object properties
    for prop in onto.object_properties():
        prop.name = prefix + prop.name

    # Rename data properties
    for prop in onto.data_properties():
        prop.name = prefix + prop.name

    # Rename annotation properties
    #for prop in onto.annotation_properties():
    #    prop.name = prefix + prop.name

    # Save the updated ontology
    onto.save(file=output_path, format="rdfxml")

    print(f"Ontology saved to {output_path} with prefix '{prefix}' added to all classes and properties.")




In [ ]:
path = './POKIMON_V1/new_pok.owlaaaaaaaaaaa'
add_prefix_to_ontology(path, path, 'POK:')

Ontology saved to ./POKIMON_V1/new_pok.owl with prefix 'POK:' added to all classes and properties.
